# Berlin Case Study: Crimes in 2012-2019

## The Problem
The city of Berlin hosts a large population (~3.9 million people) and one of the greatest concentrations of expats in Europe. As more and more people from all over the world move to Berlin — drawn by its economy, vibrant lifestyle, and cultural diversity — the city council needs to take a careful look at crime patterns across the city: understand where and how they are evolving, plan targeted interventions, and advise newcomers on what to be aware of before they arrive.

## Questions Overview
1. Which 3 districts have become more dangerous over time, and which 3 have improved?
2. Is the ratio of violent crime (robbery, assault, injury) to property crime (theft, burglary, damage) constant, or does it shift across districts?
3. Where are the crime "hot spots" that consistently rank in the top 3 across multiple crime categories simultaneously?
4. Which are the 3 districts with the lowest arson-to-fire ratio?
5. Are drug crimes a leading indicator for other violent crimes in a location?
6. Which locations are generating disproportionate crime relative to their district average?
7. How do crime rates differ between East and West Berlin districts?
8. Which are the top 3 districts that should get targeted enforcement for bike theft? Is this a seasonal or rather a structural crime? 
9. What is the relationship between graffiti/damage and more serious crimes in the same location over time?
10. Which are the 3 districts that show the greatest volatility year-to-year?

## Imports

In [1]:
import pandas as pd
import seaborn as sns
import sqlalchemy as sa

## Data Preprocessing

### Data preparation

In [2]:
df: pd.DataFrame = pd.read_csv("data/raw/berlin_crimes_2012_to_2019.csv")

In [3]:
df.head()

,Year,District,Code,Location,Robbery,Street_robbery,Injury,Agg_assault,Threat,Theft,Car,From_car,Bike,Burglary,Fire,Arson,Damage,Graffiti,Drugs,Local
0,2012,Mitte,10111,Tiergarten Süd,70,46,586,194,118,2263,18,328,120,68,16,4,273,26,171,1032
1,2012,Mitte,10112,Regierungsviertel,65,29,474,123,142,3203,10,307,170,37,10,4,380,124,98,870
2,2012,Mitte,10113,Alexanderplatz,242,136,1541,454,304,8988,81,792,822,275,49,27,1538,522,435,3108
3,2012,Mitte,10114,Brunnenstraße Süd,52,25,254,60,66,1916,86,192,396,131,14,5,428,122,213,752
4,2012,Mitte,10221,Moabit West,130,51,629,185,199,2470,94,410,325,161,42,22,516,64,259,1403


In [5]:
df.describe().round(2)

,Year,Code,Robbery,Street_robbery,Injury,Agg_assault,Threat,Theft,Car,From_car,Bike,Burglary,Fire,Arson,Damage,Graffiti,Drugs,Local
count,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00
mean,2015.50,67022.79,34.23,18.74,276.33,68.75,92.58,1492.31,42.51,215.28,197.71,69.49,15.99,6.28,281.58,62.88,97.86,662.42
std,2.29,34813.75,37.09,22.17,243.70,71.11,68.46,1364.44,28.71,150.03,178.70,57.87,12.68,5.19,203.01,62.29,174.80,534.79
min,2012.00,10111.00,0.00,0.00,0.00,0.00,0.00,17.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,10.00
25%,2013.75,40101.00,10.00,5.00,108.00,22.00,42.00,639.75,22.00,109.00,76.00,28.00,7.00,3.00,133.00,20.00,18.00,269.25
50%,2015.50,70151.50,22.00,11.00,204.50,44.00,75.00,1100.00,37.00,186.00,143.00,59.00,13.00,5.00,244.00,45.00,40.00,553.50
75%,2017.25,90520.00,42.00,23.00,361.00,86.00,124.00,2019.75,57.00,291.00,286.00,96.00,22.00,9.00,382.00,87.00,86.00,870.25
max,2019.00,129900.00,242.00,169.00,1966.00,500.00,420.00,12479.00,197.00,876.00,1288.00,446.00,74.00,31.00,1538.00,530.00,1949.00,3813.00


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Year            1200 non-null   int64
 1   District        1200 non-null   str  
 2   Code            1200 non-null   int64
 3   Location        1200 non-null   str  
 4   Robbery         1200 non-null   int64
 5   Street_robbery  1200 non-null   int64
 6   Injury          1200 non-null   int64
 7   Agg_assault     1200 non-null   int64
 8   Threat          1200 non-null   int64
 9   Theft           1200 non-null   int64
 10  Car             1200 non-null   int64
 11  From_car        1200 non-null   int64
 12  Bike            1200 non-null   int64
 13  Burglary        1200 non-null   int64
 14  Fire            1200 non-null   int64
 15  Arson           1200 non-null   int64
 16  Damage          1200 non-null   int64
 17  Graffiti        1200 non-null   int64
 18  Drugs           1200 non-null   int64
 

This dataset contains information about different types of crimes (robbery, assault, arson, drugs...) registered across the various Berlin districts and postal areas. The data available covers the 2012 to 2019 time period. Aside from geographical information (district and location), all remaining data points are numeric.

The data is already well organized and all values are present. We can create our database and upload it as is.

### Database creation

In [7]:
engine: sa.Engine = sa.create_engine("sqlite:///data/db/berlin_crimes.db")

df.to_sql(
    name="berlin_crimes",
    con=engine,
    if_exists="replace",
    index=False,
    dtype={
        "District": sa.VARCHAR,
        "Location": sa.VARCHAR,
        "Year": sa.INTEGER,
        "Robbery": sa.INTEGER,
        "Injury": sa.INTEGER,
        "Agg_assault" : sa.INTEGER,
        "Street_robbery" : sa.INTEGER,
        "Threat" : sa.INTEGER,
        "Theft" : sa.INTEGER,
        "Car" : sa.INTEGER,
        "From_car" : sa.INTEGER,
        "Bike" : sa.INTEGER,
        "Burglary" : sa.INTEGER,
        "Fire" : sa.INTEGER,
        "Arson" : sa.INTEGER,
        "Damage" : sa.INTEGER,
        "Graffiti" : sa.INTEGER,
        "Drugs" : sa.INTEGER,
        "Local" : sa.INTEGER,
    }
)

1200

In [8]:
# Check db
inspector: sa.Inspector = sa.inspect(engine)
db_cols = inspector.get_columns("berlin_crimes")

print("\n── Schema ──────────────────")
for col in db_cols:
    print(f"  {col['name']:15s}  {col['type']}")


── Schema ──────────────────
  Year             INTEGER
  District         VARCHAR
  Code             BIGINT
  Location         VARCHAR
  Robbery          INTEGER
  Street_robbery   INTEGER
  Injury           INTEGER
  Agg_assault      INTEGER
  Threat           INTEGER
  Theft            INTEGER
  Car              INTEGER
  From_car         INTEGER
  Bike             INTEGER
  Burglary         INTEGER
  Fire             INTEGER
  Arson            INTEGER
  Damage           INTEGER
  Graffiti         INTEGER
  Drugs            INTEGER
  Local            INTEGER


In [9]:
# Check data
db_first_rows: pd.DataFrame = pd.read_sql(
    sa.text(
        "SELECT Year, District, Code, Location, Robbery" \
        "FROM berlin_crimes" \
        "LIMIT 5;"
    ),
    con=engine.connect()
)
display(db_first_rows.head())

DatabaseError: Execution failed on sql 'SELECT Year, District, Code, Location, RobberyFROM berlin_crimesLIMIT 5;': (sqlite3.OperationalError) near "5": syntax error
[SQL: SELECT Year, District, Code, Location, RobberyFROM berlin_crimesLIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## A Deep Dive

### 1. Which districts have become more dangerous over time, and which have improved?